# Skin Lesion Classification — Full Pipeline (No File Uploads)

One notebook, **no `files.upload()` prompts anywhere**. Everything — Kaggle auth, data
download, training all 8 backbones (Table 1), classical classifiers on deep features
(Table 2), computational efficiency profiling (Table 3), and the final Word report — runs
automatically and the finished `.docx` downloads itself at the end.

**Before you run:** edit the `KAGGLE_USERNAME` / `KAGGLE_KEY` in the cell right below
(get them from kaggle.com -> profile picture -> Settings -> API -> Create New Token; open
the downloaded `kaggle.json` in a text editor to copy the two values). This is the only
edit needed.

**Then:** Runtime -> Change runtime type -> GPU -> Runtime -> Run all.


## 1. Kaggle credentials (edit these two values, no upload needed)

In [1]:
KAGGLE_USERNAME = "your_kaggle_username"   # <-- edit me
KAGGLE_KEY      = "your_kaggle_api_key"    # <-- edit me

import os, json as _json
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    _json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY
assert KAGGLE_USERNAME != "your_kaggle_username", "Edit KAGGLE_USERNAME/KAGGLE_KEY above before running."
print("Kaggle credentials written.")


AssertionError: Edit KAGGLE_USERNAME/KAGGLE_KEY above before running.

## 2. Setup

In [3]:
!pip -q install kaggle python-docx thop xgboost --upgrade

import time, copy, random, shutil
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from thop import profile as thop_profile

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.4 MB/s eta 0:00:00
Device: cuda


## 3. Download the Kaggle dataset (automatic, no upload)

In [2]:
!kaggle datasets download -d nodoubttome/skin-cancer9-classesisic -p /content/data --unzip

DATA_ROOT = None
for root, dirs, _ in os.walk("/content/data"):
    if "Train" in dirs and "Test" in dirs:
        DATA_ROOT = root
        break
assert DATA_ROOT, "Could not locate Train/Test folders - check the unzip output above."
print("Dataset root:", DATA_ROOT)
print("Classes found:", sorted(os.listdir(os.path.join(DATA_ROOT, "Train"))))


Dataset URL: https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic
License(s): other
100% 786M/786M [00:19<00:00, 41.9MB/s]

Dataset root: /content/data/Skin cancer ISIC The International Skin Imaging Collaboration
Classes found: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


## 4. Configuration

In [4]:
# All 9 classes normally available in this dataset:
# actinic keratosis, basal cell carcinoma, dermatofibroma, melanoma,
# nevus, pigmented benign keratosis, seborrheic keratosis,
# squamous cell carcinoma, vascular lesion
CLASSES = ["melanoma", "nevus", "basal cell carcinoma", "pigmented benign keratosis"]

IMG_SIZE    = 224
BATCH_SIZE  = 32
HEAD_EPOCHS = 5
FT_EPOCHS   = 15     # lower than the reference paper's 200 to keep this Colab-friendly; raise if you have time
LR_HEAD     = 1e-3
LR_FT       = 1e-4
NUM_CLASSES = len(CLASSES)
MODEL_NAMES_TABLE1 = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50",
                      "ResNet101", "DenseNet121", "EfficientNet-B0"]
MODEL_NAMES_TABLE3 = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50",
                      "DenseNet121", "EfficientNet-B0"]
print("Training on classes:", CLASSES)


Training on classes: ['melanoma', 'nevus', 'basal cell carcinoma', 'pigmented benign keratosis']


## 5. Stratified train / val / test split

In [5]:
def collect_files(split):
    rows = []
    split_dir = os.path.join(DATA_ROOT, split)
    for cls in CLASSES:
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            raise FileNotFoundError(f"Class folder not found: {cls_dir}")
        for fn in os.listdir(cls_dir):
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                rows.append((os.path.join(cls_dir, fn), cls))
    return pd.DataFrame(rows, columns=["path", "label"])

train_full = collect_files("Train")
test_df    = collect_files("Test")

train_df, val_df = train_test_split(
    train_full, test_size=0.15, stratify=train_full["label"], random_state=SEED
)

class_to_idx = {c: i for i, c in enumerate(CLASSES)}
for d in (train_df, val_df, test_df):
    d["y"] = d["label"].map(class_to_idx)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print(train_df["label"].value_counts())


Train: 1388 Val: 245 Test: 64
label
pigmented benign keratosis    393
melanoma                      372
basal cell carcinoma          320
nevus                         303
Name: count, dtype: int64


## 6. Dataset & DataLoaders

In [6]:
class LesionDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = self.transform(img)
        return img, row["y"]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), scale=(0.8, 1.2), shear=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = LesionDataset(train_df, train_tf)
val_ds   = LesionDataset(val_df, eval_tf)
test_ds  = LesionDataset(test_df, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 7. Model zoo

In [7]:
def build_model(name, num_classes):
    """Returns an ImageNet-pretrained model with its head replaced for num_classes."""
    if name == "AlexNet":
        m = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        head_params = m.classifier[6].parameters()
    elif name == "VGG16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        head_params = m.classifier[6].parameters()
    elif name == "VGG19":
        m = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        head_params = m.classifier[6].parameters()
    elif name == "ResNet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head_params = m.fc.parameters()
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head_params = m.fc.parameters()
    elif name == "ResNet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head_params = m.fc.parameters()
    elif name == "DenseNet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        head_params = m.classifier.parameters()
    elif name == "EfficientNet-B0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        head_params = m.classifier[1].parameters()
    else:
        raise ValueError(name)
    return m, list(head_params)


## 8. Training & evaluation helpers

In [8]:
def set_backbone_trainable(model, trainable):
    for p in model.parameters():
        p.requires_grad = trainable

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for x, y in loader:
        x = x.to(device)
        out = model(x)
        all_logits.append(out.cpu().numpy())
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

def compute_metrics(logits, y_true):
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0) * 100
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0) * 100
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro") * 100
    except ValueError:
        auc = float("nan")
    return {"Accuracy (%)": acc, "Precision (%)": prec, "Recall (%)": rec,
            "F1-Score (%)": f1, "AUC (%)": auc}

def fine_tune_model(name, epochs_head=HEAD_EPOCHS, epochs_ft=FT_EPOCHS):
    model, head_params = build_model(name, NUM_CLASSES)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    set_backbone_trainable(model, False)
    for p in head_params:
        p.requires_grad = True
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD)
    for ep in range(epochs_head):
        loss = train_one_epoch(model, train_loader, opt, criterion)
        print(f"  [{name}] head epoch {ep+1}/{epochs_head} loss={loss:.4f}")

    set_backbone_trainable(model, True)
    opt = torch.optim.Adam(model.parameters(), lr=LR_FT)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    best_val = float("inf"); best_state = copy.deepcopy(model.state_dict())
    for ep in range(epochs_ft):
        loss = train_one_epoch(model, train_loader, opt, criterion)
        val_logits, val_y = get_logits(model, val_loader)
        val_loss = criterion(torch.tensor(val_logits), torch.tensor(val_y)).item()
        sched.step(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
        print(f"  [{name}] ft epoch {ep+1}/{epochs_ft} train_loss={loss:.4f} val_loss={val_loss:.4f}")

    model.load_state_dict(best_state)
    return model


## 9. Table 1 — train + evaluate all 8 backbones\n⚠️ Long-running cell. State dicts are kept in memory for reuse in Tables 2 and 3.

In [9]:
table1_results = {}
trained_state_dicts = {}

for name in MODEL_NAMES_TABLE1:
    print(f"=== Training {name} ===")
    model = fine_tune_model(name)
    test_logits, test_y = get_logits(model, test_loader)
    metrics = compute_metrics(test_logits, test_y)
    table1_results[name] = metrics
    print(f"  {name} test metrics: {metrics}")
    trained_state_dicts[name] = copy.deepcopy(model.state_dict())
    del model
    torch.cuda.empty_cache()

table1_df = pd.DataFrame(table1_results).T
table1_df.index.name = "Model"
table1_df = table1_df.round(2)
table1_df.to_csv("table1_results.csv")
table1_df


=== Training AlexNet ===
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 173MB/s]


  [AlexNet] head epoch 1/5 loss=1.1243
  [AlexNet] head epoch 2/5 loss=0.9218
  [AlexNet] head epoch 3/5 loss=0.9075
  [AlexNet] head epoch 4/5 loss=0.8938
  [AlexNet] head epoch 5/5 loss=0.8543
  [AlexNet] ft epoch 1/15 train_loss=0.8327 val_loss=0.6958
  [AlexNet] ft epoch 2/15 train_loss=0.6456 val_loss=0.6451
  [AlexNet] ft epoch 3/15 train_loss=0.5677 val_loss=0.6948
  [AlexNet] ft epoch 4/15 train_loss=0.5015 val_loss=0.5262
  [AlexNet] ft epoch 5/15 train_loss=0.4473 val_loss=0.5800
  [AlexNet] ft epoch 6/15 train_loss=0.4211 val_loss=0.7323
  [AlexNet] ft epoch 7/15 train_loss=0.4630 val_loss=0.5411
  [AlexNet] ft epoch 8/15 train_loss=0.3708 val_loss=0.5884
  [AlexNet] ft epoch 9/15 train_loss=0.3162 val_loss=0.5849
  [AlexNet] ft epoch 10/15 train_loss=0.2827 val_loss=0.6622
  [AlexNet] ft epoch 11/15 train_loss=0.2708 val_loss=0.5925
  [AlexNet] ft epoch 12/15 train_loss=0.2499 val_loss=0.5971
  [AlexNet] ft epoch 13/15 train_loss=0.2114 val_loss=0.6435
  [AlexNet] ft epoch 

100%|██████████| 528M/528M [00:09<00:00, 60.2MB/s]


  [VGG16] head epoch 1/5 loss=1.2752
  [VGG16] head epoch 2/5 loss=1.1446
  [VGG16] head epoch 3/5 loss=1.0727
  [VGG16] head epoch 4/5 loss=1.1002
  [VGG16] head epoch 5/5 loss=1.0357
  [VGG16] ft epoch 1/15 train_loss=1.1920 val_loss=0.9245
  [VGG16] ft epoch 2/15 train_loss=0.8888 val_loss=0.7097
  [VGG16] ft epoch 3/15 train_loss=0.7862 val_loss=0.7563
  [VGG16] ft epoch 4/15 train_loss=0.6398 val_loss=0.5560
  [VGG16] ft epoch 5/15 train_loss=0.5833 val_loss=0.5549
  [VGG16] ft epoch 6/15 train_loss=0.5075 val_loss=0.8703
  [VGG16] ft epoch 7/15 train_loss=0.4870 val_loss=0.5513
  [VGG16] ft epoch 8/15 train_loss=0.4494 val_loss=1.0929
  [VGG16] ft epoch 9/15 train_loss=0.4266 val_loss=0.5194
  [VGG16] ft epoch 10/15 train_loss=0.3915 val_loss=0.5366
  [VGG16] ft epoch 11/15 train_loss=0.4745 val_loss=0.6960
  [VGG16] ft epoch 12/15 train_loss=0.3829 val_loss=0.5354
  [VGG16] ft epoch 13/15 train_loss=0.3558 val_loss=0.5981
  [VGG16] ft epoch 14/15 train_loss=0.2819 val_loss=0.698

...

100%|██████████| 30.8M/30.8M [00:00<00:00, 154MB/s]


  [DenseNet121] head epoch 1/5 loss=1.2586
  [DenseNet121] head epoch 2/5 loss=1.0482
  [DenseNet121] head epoch 3/5 loss=0.9282
  [DenseNet121] head epoch 4/5 loss=0.8712
  [DenseNet121] head epoch 5/5 loss=0.8392
  [DenseNet121] ft epoch 1/15 train_loss=0.6869 val_loss=0.5882
  [DenseNet121] ft epoch 2/15 train_loss=0.4626 val_loss=0.5645
  [DenseNet121] ft epoch 3/15 train_loss=0.3936 val_loss=0.5061
  [DenseNet121] ft epoch 4/15 train_loss=0.3576 val_loss=0.4839
  [DenseNet121] ft epoch 5/15 train_loss=0.3021 val_loss=0.4705
  [DenseNet121] ft epoch 6/15 train_loss=0.2212 val_loss=0.5023
  [DenseNet121] ft epoch 7/15 train_loss=0.1955 val_loss=0.5546
  [DenseNet121] ft epoch 8/15 train_loss=0.1402 val_loss=0.5409
  [DenseNet121] ft epoch 9/15 train_loss=0.1720 val_loss=0.5731
  [DenseNet121] ft epoch 10/15 train_loss=0.1384 val_loss=0.5797
  [DenseNet121] ft epoch 11/15 train_loss=0.1139 val_loss=0.5245
  [DenseNet121] ft epoch 12/15 train_loss=0.0909 val_loss=0.5874
  [DenseNet121

100%|██████████| 20.5M/20.5M [00:00<00:00, 163MB/s]


  [EfficientNet-B0] head epoch 1/5 loss=1.1804
  [EfficientNet-B0] head epoch 2/5 loss=0.9450
  [EfficientNet-B0] head epoch 3/5 loss=0.8760
  [EfficientNet-B0] head epoch 4/5 loss=0.8094
  [EfficientNet-B0] head epoch 5/5 loss=0.8016
  [EfficientNet-B0] ft epoch 1/15 train_loss=0.6784 val_loss=0.6546
  [EfficientNet-B0] ft epoch 2/15 train_loss=0.5332 val_loss=0.5791
  [EfficientNet-B0] ft epoch 3/15 train_loss=0.4377 val_loss=0.5646
  [EfficientNet-B0] ft epoch 4/15 train_loss=0.3916 val_loss=0.4865
  [EfficientNet-B0] ft epoch 5/15 train_loss=0.3778 val_loss=0.4564
  [EfficientNet-B0] ft epoch 6/15 train_loss=0.3052 val_loss=0.4738
  [EfficientNet-B0] ft epoch 7/15 train_loss=0.2554 val_loss=0.5077
  [EfficientNet-B0] ft epoch 8/15 train_loss=0.2278 val_loss=0.5020
  [EfficientNet-B0] ft epoch 9/15 train_loss=0.2210 val_loss=0.5139
  [EfficientNet-B0] ft epoch 10/15 train_loss=0.1812 val_loss=0.4642
  [EfficientNet-B0] ft epoch 11/15 train_loss=0.1470 val_loss=0.4692
  [EfficientNet

,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
Model,,,,,
AlexNet,65.62,66.98,65.62,62.23,89.42
VGG16,73.44,78.45,73.44,71.48,90.85
VGG19,70.31,56.60,70.31,61.71,85.40
ResNet18,65.62,71.21,65.62,62.45,88.96
ResNet50,75.00,80.43,75.00,73.19,88.93
ResNet101,65.62,75.55,65.62,62.35,88.51
DenseNet121,70.31,78.76,70.31,66.44,91.50
EfficientNet-B0,70.31,78.89,70.31,67.67,92.74


## 10. Table 2 — classical classifiers on deep features\nReuses the best Table-1 backbone as a frozen feature extractor (no retraining).

In [10]:
best_model_name = table1_df["Accuracy (%)"].astype(float).idxmax()
print("Best backbone for feature extraction:", best_model_name)

best_model, _ = build_model(best_model_name, NUM_CLASSES)
best_model.load_state_dict(trained_state_dicts[best_model_name])
best_model = best_model.to(device).eval()

feature_extractor = copy.deepcopy(best_model)
if best_model_name in ("AlexNet", "VGG16", "VGG19"):
    feature_extractor.classifier = feature_extractor.classifier[:-1]
elif best_model_name in ("ResNet18", "ResNet50", "ResNet101"):
    feature_extractor.fc = nn.Identity()
elif best_model_name == "DenseNet121":
    feature_extractor.classifier = nn.Identity()
elif best_model_name == "EfficientNet-B0":
    feature_extractor.classifier[1] = nn.Identity()
feature_extractor.eval()

@torch.no_grad()
def extract_features(loader):
    feats, ys = [], []
    for x, y in loader:
        x = x.to(device)
        f = feature_extractor(x)
        f = torch.flatten(f, 1)
        feats.append(f.cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(feats), np.concatenate(ys)

X_train_feat, y_train_feat = extract_features(train_loader)
X_test_feat, y_test_feat   = extract_features(test_loader)
print("Feature shape:", X_train_feat.shape)


Best backbone for feature extraction: ResNet50
Feature shape: (1388, 2048)


In [11]:
def eval_sklearn_clf(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0) * 100
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0) * 100
    try:
        if hasattr(clf, "predict_proba"):
            proba = clf.predict_proba(X_test)
        elif hasattr(clf, "decision_function"):
            from scipy.special import softmax
            proba = softmax(clf.decision_function(X_test), axis=1)
        else:
            proba = None
        auc = roc_auc_score(y_test, proba, multi_class="ovr", average="macro") * 100 if proba is not None else float("nan")
    except ValueError:
        auc = float("nan")
    return {"Accuracy (%)": acc, "Precision (%)": prec, "Recall (%)": rec,
            "F1-Score (%)": f1, "AUC (%)": auc}

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": SVC(kernel="linear", probability=True, random_state=SEED),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=SEED),
}

table2_results = {}
for cname, clf in classifiers.items():
    print(f"Fitting {cname} ...")
    table2_results[cname] = eval_sklearn_clf(clf, X_train_feat, y_train_feat, X_test_feat, y_test_feat)

table2_df = pd.DataFrame(table2_results).T
table2_df.insert(0, "Feature Extractor", "Deep Features")
table2_df.index.name = "Classifier"
table2_df.iloc[:, 1:] = table2_df.iloc[:, 1:].round(2)
table2_df.to_csv("table2_results.csv")
table2_df


Fitting Logistic Regression ...
Fitting Decision Tree ...
Fitting Random Forest ...
Fitting K-Nearest Neighbors (KNN) ...
Fitting Linear SVM ...
Fitting RBF-SVM ...
Fitting XGBoost ...


,Feature Extractor,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
Classifier,,,,,,
Logistic Regression,Deep Features,73.44,78.08,73.44,71.98,85.97
Decision Tree,Deep Features,56.25,55.76,56.25,53.42,70.83
Random Forest,Deep Features,76.56,83.09,76.56,72.98,90.32
K-Nearest Neighbors (KNN),Deep Features,67.19,70.37,67.19,65.57,85.79
Linear SVM,Deep Features,73.44,78.08,73.44,71.98,86.52
RBF-SVM,Deep Features,73.44,79.24,73.44,71.66,91.70
XGBoost,Deep Features,76.56,82.11,76.56,74.70,89.42


## 11. Table 3 — computational efficiency\nReuses the already-trained Table-1 weights (no retraining) — just profiles params/size/FLOPs/latency.

In [12]:
@torch.no_grad()
def benchmark_model(name, n_warmup=10, n_runs=50):
    model, _ = build_model(name, NUM_CLASSES)
    model.load_state_dict(trained_state_dicts[name])
    model = model.to(device).eval()

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    flops, params = thop_profile(copy.deepcopy(model), inputs=(dummy,), verbose=False)

    tmp_path = f"/content/{name}_tmp.pt"
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / 1e6
    os.remove(tmp_path)

    for _ in range(n_warmup):
        model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_runs):
        model(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.time() - start) / n_runs * 1000

    del model
    torch.cuda.empty_cache()
    return {
        "Parameters (M)": params / 1e6,
        "Model Size (MB)": size_mb,
        "FLOPs (G)": flops / 1e9,
        "Inference Time (ms)": elapsed_ms,
        "Accuracy (%)": float(table1_df.loc[name, "Accuracy (%)"]),
    }

table3_results = {name: benchmark_model(name) for name in MODEL_NAMES_TABLE3}
table3_df = pd.DataFrame(table3_results).T
table3_df.index.name = "Model"
table3_df = table3_df.round(3)
table3_df.to_csv("table3_results.csv")
table3_df


,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
Model,,,,,
AlexNet,57.020,228.087,0.710,2.111,65.62
VGG16,134.277,537.120,15.466,9.013,73.44
VGG19,139.587,558.361,19.628,10.741,70.31
ResNet18,11.179,44.794,1.824,2.502,65.62
ResNet50,23.516,94.385,4.132,6.023,75.00
DenseNet121,6.958,28.443,2.896,17.289,70.31
EfficientNet-B0,4.013,16.353,0.414,10.324,70.31


## 12. Build the Word report from scratch and auto-download\nNo template upload — the whole `.docx` (methodology notes + all three filled tables) is generated in code and downloads automatically.

In [14]:
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from google.colab import files

doc = Document()
style = doc.styles["Normal"]
style.font.name = "Times New Roman"
style.font.size = Pt(11)

def add_heading_text(text, bold=True, italic=False):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold
    run.italic = italic
    run.font.name = "Times New Roman"
    return p

def add_body_text(text):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.name = "Times New Roman"
    return p

add_heading_text("Methodology Notes")
add_body_text(
    f"Dataset: Kaggle ‘Skin Cancer 9 Classes ISIC’ dataset, restricted to a 4-class "
    f"subset: {', '.join(c.title() for c in CLASSES)}. These classes mirror the highest-prevalence "
    f"and most clinically consequential classes used in the underlying HAM10000-derived literature "
    f"(e.g., Rahman & Ami, 2020, bioRxiv 860700), spanning malignant and benign lesions."
)
add_body_text(
    "Approach: following the transfer-learning / fine-tuning strategy used in the reference paper "
    "(ImageNet-pretrained backbones, frozen-then-unfrozen fine-tuning, Adam optimizer with a "
    "reduce-on-plateau schedule), Table 1 compares eight ImageNet-pretrained CNN backbones fine-tuned "
    "end-to-end. Table 2 freezes the best backbone as a fixed feature extractor and compares classical "
    "ML classifiers trained on the extracted deep features. Table 3 profiles the computational cost of "
    "each backbone (parameter count, on-disk size, FLOPs, and inference latency) alongside its accuracy."
)
add_body_text(f"Best backbone used for Table 2 feature extraction: {best_model_name}.")

def add_table(df, title, index_label, extra_index_cols=None):
    add_heading_text(title, bold=False)
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER

    cols = ([index_label] if not extra_index_cols else extra_index_cols) + list(df.columns)
    n_cols = len(cols)
    table = doc.add_table(rows=1, cols=n_cols)
    table.style = "Table Grid"

    hdr = table.rows[0].cells
    for i, col_name in enumerate(cols):
        hdr[i].text = col_name
        hdr[i].paragraphs[0].runs[0].bold = True

    for idx, row in df.iterrows():
        cells = table.add_row().cells
        if extra_index_cols:
            cells[0].text = str(row[extra_index_cols[0]])
            cells[1].text = str(idx)
            start = 2
            values = row.drop(labels=[extra_index_cols[0]])
        else:
            cells[0].text = str(idx)
            start = 1
            values = row
        for j, val in enumerate(values):
            cells[start + j].text = f"{val:.2f}" if isinstance(val, (int, float)) else str(val)
    doc.add_paragraph()

add_table(table1_df, "Table 1. Comparison of Transfer Learning Models", "Model")
add_table(table2_df, "Table 2. Comparison of Different Classifiers",
          "Classifier", extra_index_cols=["Feature Extractor"])
add_table(table3_df, "Table 3. Computational Efficiency Comparison", "Model")

out_path = "/content/Task_01_completed.docx"
doc.save(out_path)
print("Saved:", out_path)
files.download(out_path)


Saved: /content/Task_01_completed.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes
- The only manual step is pasting your Kaggle `username`/`key` into section 1 — everything else (download, training, evaluation, the finished Word doc) runs unattended.
- `FT_EPOCHS = 15` is a Colab-friendly budget (the reference paper trains 200 epochs) — raise it in section 4 if you have more GPU time.
- Change `CLASSES` in section 4 to use a different 4-of-9 class subset.
- If Colab disconnects mid-run, each table is also saved as a CSV (`table1_results.csv`, `table2_results.csv`, `table3_results.csv`) so you don't have to redo everything — reload with `pd.read_csv(..., index_col=0)` and re-run from section 12.
